# LightGBM Model for Delinquency Prediction

Uses the **top 50 selected features** from the comprehensive feature selection pipeline.

- Train / Validation / Test split (60/20/20)
- LightGBM gradient boosting classifier
- Handles class imbalance with `scale_pos_weight`
- Records training time, scoring time, and model performance

In [65]:
import sys
import pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix,
    precision_recall_curve, average_precision_score
)
import matplotlib.pyplot as plt
import time
import warnings
import importlib
warnings.filterwarnings('ignore')

import scripts.feature_selection as _fs
importlib.reload(_fs)
import scripts.model_data as _md
importlib.reload(_md)
from scripts.model_data import load_and_split


## 1. Load Data & Selected Features

In [66]:
# ── Configuration ────────────────────────────────────────────────
N_FEATURES = 50  # ← change this to use a different number of top-ranked features
FEATURES_FILE = "filtered_features_consensus_ordered.csv"

## 2. Prepare Data & Train/Val/Test Split

In [67]:
# Load features and perform 60/20/20 stratified split
X_train, X_val, X_test, y_train, y_val, y_test, available_features, features_df = \
    load_and_split(n_features=N_FEATURES, features_filename=FEATURES_FILE)

Features  : 50 (top-50)
Samples   : 10754  |  DQ rate: 8.18%
Train     : 6452  (8.18% positive)
Val       : 2151  (8.18% positive)
Test      : 2151  (8.18% positive)


## 3. Train LightGBM Model

In [68]:
# Class imbalance weight
n_neg = np.sum(y_train == 0)
n_pos = np.sum(y_train == 1)
scale_pos_weight = n_neg / n_pos
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

# Create LightGBM datasets
train_data = lgb.Dataset(X_train, label=y_train, feature_name=available_features)
val_data = lgb.Dataset(X_val, label=y_val, feature_name=available_features, reference=train_data)

# LightGBM parameters
params = {
    'objective': 'binary',
    'metric': ['auc', 'binary_logloss'],
    'boosting_type': 'gbdt',
    'scale_pos_weight': scale_pos_weight,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'verbose': -1,
    'seed': 42,
}

# Train with early stopping
print('\nTraining LightGBM...')
training_start_time = time.time()

evals_result = {}

callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=True),
    lgb.log_evaluation(period=50),
    lgb.record_evaluation(evals_result),
]

model = lgb.train(
    params,
    train_data,
    num_boost_round=1000,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'val'],
    callbacks=callbacks,
)

training_end_time = time.time()
training_time = training_end_time - training_start_time

print(f'\nTraining completed in {training_time:.2f}s')
print(f'Best iteration: {model.best_iteration}')
print(f'Best val AUC: {model.best_score["val"]["auc"]:.4f}')


scale_pos_weight: 11.22

Training LightGBM...
Training until validation scores don't improve for 50 rounds
[50]	train's auc: 0.982699	train's binary_logloss: 0.273981	val's auc: 0.77645	val's binary_logloss: 0.365598
Early stopping, best iteration is:
[2]	train's auc: 0.870136	train's binary_logloss: 0.259412	val's auc: 0.7533	val's binary_logloss: 0.271108

Training completed in 0.50s
Best iteration: 2
Best val AUC: 0.7533


## 5. Evaluation on All Splits

In [69]:
def evaluate(model, X, y_true, split_name):
    """Evaluate model and print metrics."""
    scoring_start = time.time()
    
    # Predict probabilities
    y_proba = model.predict(X, num_iteration=model.best_iteration)
    
    scoring_time = time.time() - scoring_start
    
    # AUC
    auc = roc_auc_score(y_true, y_proba)
    ap = average_precision_score(y_true, y_proba)
    
    # Binary predictions at 0.5 threshold
    y_pred = (y_proba >= 0.5).astype(int)
    
    print(f'\n{"="*50}')
    print(f'{split_name} Results')
    print(f'{"="*50}')
    print(f'ROC-AUC: {auc:.4f}')
    print(f'Average Precision: {ap:.4f}')
    print(f'Scoring Time: {scoring_time:.4f} seconds')
    print(f'\nClassification Report:')
    print(classification_report(y_true, y_pred, target_names=['No DQ', 'DQ']))
    print(f'Confusion Matrix:')
    print(confusion_matrix(y_true, y_pred))
    
    return y_proba, auc, ap, scoring_time

train_preds, train_auc, train_ap, train_time = evaluate(model, X_train, y_train, 'TRAIN')
val_preds, val_auc, val_ap, val_time = evaluate(model, X_val, y_val, 'VALIDATION')
test_preds, test_auc, test_ap, test_time = evaluate(model, X_test, y_test, 'TEST')


TRAIN Results
ROC-AUC: 0.8701
Average Precision: 0.3071
Scoring Time: 0.0010 seconds

Classification Report:
              precision    recall  f1-score   support

       No DQ       0.92      1.00      0.96      5924
          DQ       0.00      0.00      0.00       528

    accuracy                           0.92      6452
   macro avg       0.46      0.50      0.48      6452
weighted avg       0.84      0.92      0.88      6452

Confusion Matrix:
[[5924    0]
 [ 528    0]]

VALIDATION Results
ROC-AUC: 0.7533
Average Precision: 0.2187
Scoring Time: 0.0000 seconds

Classification Report:
              precision    recall  f1-score   support

       No DQ       0.92      1.00      0.96      1975
          DQ       0.00      0.00      0.00       176

    accuracy                           0.92      2151
   macro avg       0.46      0.50      0.48      2151
weighted avg       0.84      0.92      0.88      2151

Confusion Matrix:
[[1975    0]
 [ 176    0]]

TEST Results
ROC-AUC: 0.7467
A

## 7. Summary

In [70]:
print('LightGBM Model Summary')
print('=' * 50)
print(f'Boosting type: GBDT')
print(f'Num leaves: {params["num_leaves"]}')
print(f'Learning rate: {params["learning_rate"]}')
print(f'Best iteration: {model.best_iteration}')
print(f'Features used: {len(available_features)}')
print(f'')
print(f'Training Time: {training_time:.2f}s ({training_time/60:.2f} min)')
print(f'')
print(f'{"Split":<12} {"ROC-AUC":<12} {"Avg Precision":<16} {"Scoring Time":<14}')
print(f'{"-"*54}')
print(f'{"Train":<12} {train_auc:<12.4f} {train_ap:<16.4f} {train_time:<14.4f}s')
print(f'{"Val":<12} {val_auc:<12.4f} {val_ap:<16.4f} {val_time:<14.4f}s')
print(f'{"Test":<12} {test_auc:<12.4f} {test_ap:<16.4f} {test_time:<14.4f}s')

LightGBM Model Summary
Boosting type: GBDT
Num leaves: 31
Learning rate: 0.05
Best iteration: 2
Features used: 50

Training Time: 0.50s (0.01 min)

Split        ROC-AUC      Avg Precision    Scoring Time  
------------------------------------------------------
Train        0.8701       0.3071           0.0010        s
Val          0.7533       0.2187           0.0000        s
Test         0.7467       0.2119           0.0006        s


---
# LightGBM – All Features

Same setup but using **every available feature** instead of the top-50 selection.

In [71]:
# ── Use ALL features ─────────────────────────────────────────────
X_train_all, X_val_all, X_test_all, y_train_all, y_val_all, y_test_all, all_feature_cols, _ = \
    load_and_split(use_all=True)

Features  : 241 (all)
Samples   : 12000  |  DQ rate: 8.38%
Train     : 7200  (8.39% positive)
Val       : 2400  (8.38% positive)
Test      : 2400  (8.38% positive)


In [72]:
# ── Train LightGBM (all features) ────────────────────────────────
n_neg_all = np.sum(y_train_all == 0)
n_pos_all = np.sum(y_train_all == 1)
scale_pos_weight_all = n_neg_all / n_pos_all
print(f'scale_pos_weight: {scale_pos_weight_all:.2f}')

train_data_all = lgb.Dataset(X_train_all, label=y_train_all, feature_name=all_feature_cols)
val_data_all   = lgb.Dataset(X_val_all,   label=y_val_all,   feature_name=all_feature_cols,
                              reference=train_data_all)

params_all = {
    'objective': 'binary',
    'metric': ['auc', 'binary_logloss'],
    'boosting_type': 'gbdt',
    'scale_pos_weight': scale_pos_weight_all,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'verbose': -1,
    'seed': 42,
}

print('\nTraining LightGBM (all features)...')
train_start_all = time.time()

evals_result_all = {}

model_all = lgb.train(
    params_all,
    train_data_all,
    num_boost_round=1000,
    valid_sets=[train_data_all, val_data_all],
    valid_names=['train', 'val'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(period=50),
        lgb.record_evaluation(evals_result_all),
    ],
)

training_time_all = time.time() - train_start_all
print(f'\nTraining completed in {training_time_all:.2f}s')
print(f'Best iteration: {model_all.best_iteration}')
print(f'Best val AUC: {model_all.best_score["val"]["auc"]:.4f}')


scale_pos_weight: 10.92

Training LightGBM (all features)...
Training until validation scores don't improve for 50 rounds
[50]	train's auc: 0.979796	train's binary_logloss: 0.279385	val's auc: 0.772212	val's binary_logloss: 0.366006
Early stopping, best iteration is:
[2]	train's auc: 0.871095	train's binary_logloss: 0.264632	val's auc: 0.751969	val's binary_logloss: 0.275409

Training completed in 0.66s
Best iteration: 2
Best val AUC: 0.7520


In [73]:
# ── Evaluation (all features) ────────────────────────────────────
train_preds_all, train_auc_all, train_ap_all, train_time_all = evaluate(
    model_all, X_train_all, y_train_all, 'TRAIN (all features)')
val_preds_all, val_auc_all, val_ap_all, val_time_all = evaluate(
    model_all, X_val_all, y_val_all, 'VALIDATION (all features)')
test_preds_all, test_auc_all, test_ap_all, test_time_all = evaluate(
    model_all, X_test_all, y_test_all, 'TEST (all features)')


TRAIN (all features) Results
ROC-AUC: 0.8711
Average Precision: 0.3083
Scoring Time: 0.0000 seconds

Classification Report:
              precision    recall  f1-score   support

       No DQ       0.92      1.00      0.96      6596
          DQ       0.00      0.00      0.00       604

    accuracy                           0.92      7200
   macro avg       0.46      0.50      0.48      7200
weighted avg       0.84      0.92      0.88      7200

Confusion Matrix:
[[6596    0]
 [ 604    0]]

VALIDATION (all features) Results
ROC-AUC: 0.7520
Average Precision: 0.2296
Scoring Time: 0.0000 seconds

Classification Report:
              precision    recall  f1-score   support

       No DQ       0.92      1.00      0.96      2199
          DQ       0.00      0.00      0.00       201

    accuracy                           0.92      2400
   macro avg       0.46      0.50      0.48      2400
weighted avg       0.84      0.92      0.88      2400

Confusion Matrix:
[[2199    0]
 [ 201    0]]



In [74]:
# ── Summary: All Features vs Top-50 ──────────────────────────────
print('Comparison: Top-50 Features vs All Features')
print('=' * 70)
print(f'{"Metric":<22} {"Top-50":<22} {"All Features":<22}')
print(f'{"-"*66}')
print(f'{"Num features":<22} {len(available_features):<22} {len(all_feature_cols):<22}')
print(f'{"Best iteration":<22} {model.best_iteration:<22} {model_all.best_iteration:<22}')
print(f'{"Training time (s)":<22} {training_time:<22.2f} {training_time_all:<22.2f}')
print()
print(f'{"Split":<8} {"Metric":<16} {"Top-50":<14} {"All Features":<14}')
print(f'{"-"*52}')
for split, (auc50, ap50, t50, auca, apa, ta) in {
    'Train': (train_auc, train_ap, train_time, train_auc_all, train_ap_all, train_time_all),
    'Val':   (val_auc,   val_ap,   val_time,   val_auc_all,   val_ap_all,   val_time_all),
    'Test':  (test_auc,  test_ap,  test_time,  test_auc_all,  test_ap_all,  test_time_all),
}.items():
    print(f'{split:<8} {"ROC-AUC":<16} {auc50:<14.4f} {auca:<14.4f}')
    print(f'{"":8} {"Avg Precision":<16} {ap50:<14.4f} {apa:<14.4f}')
    print(f'{"":8} {"Score time (s)":<16} {t50:<14.4f} {ta:<14.4f}')

Comparison: Top-50 Features vs All Features
Metric                 Top-50                 All Features          
------------------------------------------------------------------
Num features           50                     241                   
Best iteration         2                      2                     
Training time (s)      0.50                   0.66                  

Split    Metric           Top-50         All Features  
----------------------------------------------------
Train    ROC-AUC          0.8701         0.8711        
         Avg Precision    0.3071         0.3083        
         Score time (s)   0.0010         0.0000        
Val      ROC-AUC          0.7533         0.7520        
         Avg Precision    0.2187         0.2296        
         Score time (s)   0.0000         0.0000        
Test     ROC-AUC          0.7467         0.7468        
         Avg Precision    0.2119         0.2148        
         Score time (s)   0.0006         0.0010        

---
## 9. Regularized Model (Overfitting Fix)

Early overfitting is caused by trees that are too complex. Key changes vs the baseline:

| Parameter | Baseline | Regularized | Why |
|---|---|---|---|
| `num_leaves` | 31 | **15** | Fewer leaves → simpler trees |
| `max_depth` | -1 (unlimited) | **5** | Hard cap on tree depth |
| `min_child_samples` | 20 | **100** | Each leaf needs more data |
| `subsample` | 0.8 | **0.6** | More randomness per tree |
| `colsample_bytree` | 0.8 | **0.6** | Sample fewer features per tree |
| `reg_alpha` | 0.1 | **1.0** | L1 sparsity regularization |
| `reg_lambda` | 0.1 | **5.0** | L2 weight regularization |
| `min_split_gain` | 0 | **0.05** | Require meaningful splits only |
| `learning_rate` | 0.05 | **0.02** | Slower learning, more rounds |

In [75]:
# ── Regularized LightGBM (Top-50 features) ───────────────────────
params_reg = {
    'objective':          'binary',
    'metric':             ['auc', 'binary_logloss'],
    'boosting_type':      'gbdt',
    'scale_pos_weight':   scale_pos_weight,

    # ── tree complexity (main overfitting levers) ──
    'num_leaves':         15,     # was 31
    'max_depth':          5,      # was -1 (unlimited)
    'min_child_samples':  100,    # was 20

    # ── stochasticity ──
    'subsample':          0.6,    # was 0.8
    'colsample_bytree':   0.6,    # was 0.8

    # ── regularization ──
    'reg_alpha':          1.0,    # was 0.1
    'reg_lambda':         5.0,    # was 0.1
    'min_split_gain':     0.05,   # was 0 (no threshold)

    # ── learning ──
    'learning_rate':      0.02,   # was 0.05 (slower + more rounds)
    'verbose':            -1,
    'seed':               42,
}

evals_result_reg = {}

print('Training regularized LightGBM...')
t0_reg = time.time()

model_reg = lgb.train(
    params_reg,
    train_data,                     # same split as baseline
    num_boost_round=2000,           # more rounds at lower LR
    valid_sets=[train_data, val_data],
    valid_names=['train', 'val'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=100, verbose=True),
        lgb.log_evaluation(period=100),
        lgb.record_evaluation(evals_result_reg),
    ],
)

training_time_reg = time.time() - t0_reg
print(f'\nDone in {training_time_reg:.2f}s  |  best iteration: {model_reg.best_iteration}')
print(f'Best val AUC: {model_reg.best_score["val"]["auc"]:.4f}')


Training regularized LightGBM...
Training until validation scores don't improve for 100 rounds
[100]	train's auc: 0.906816	train's binary_logloss: 0.388822	val's auc: 0.779317	val's binary_logloss: 0.430857
Early stopping, best iteration is:
[5]	train's auc: 0.824309	train's binary_logloss: 0.269327	val's auc: 0.75732	val's binary_logloss: 0.273782

Done in 0.87s  |  best iteration: 5
Best val AUC: 0.7573


In [76]:
# ── Evaluate regularized model ────────────────────────────────────
train_preds_reg, train_auc_reg, train_ap_reg, train_time_reg = evaluate(
    model_reg, X_train, y_train, 'TRAIN (regularized)')
val_preds_reg, val_auc_reg, val_ap_reg, val_time_reg = evaluate(
    model_reg, X_val, y_val, 'VALIDATION (regularized)')
test_preds_reg, test_auc_reg, test_ap_reg, test_time_reg = evaluate(
    model_reg, X_test, y_test, 'TEST (regularized)')

print('\nBaseline vs Regularized (Top-50)')
print('=' * 58)
print(f'{"Split":<8} {"Metric":<18} {"Baseline":<14} {"Regularized":<14}')
print(f'{"-"*58}')
for split_name, (auc_b, auc_r) in {
    'Train': (train_auc,     train_auc_reg),
    'Val':   (val_auc,       val_auc_reg),
    'Test':  (test_auc,      test_auc_reg),
}.items():
    print(f'{split_name:<8} {"ROC-AUC":<18} {auc_b:<14.4f} {auc_r:<14.4f}')



TRAIN (regularized) Results
ROC-AUC: 0.8243
Average Precision: 0.2541
Scoring Time: 0.0000 seconds

Classification Report:
              precision    recall  f1-score   support

       No DQ       0.92      1.00      0.96      5924
          DQ       0.00      0.00      0.00       528

    accuracy                           0.92      6452
   macro avg       0.46      0.50      0.48      6452
weighted avg       0.84      0.92      0.88      6452

Confusion Matrix:
[[5924    0]
 [ 528    0]]

VALIDATION (regularized) Results
ROC-AUC: 0.7573
Average Precision: 0.2081
Scoring Time: 0.0000 seconds

Classification Report:
              precision    recall  f1-score   support

       No DQ       0.92      1.00      0.96      1975
          DQ       0.00      0.00      0.00       176

    accuracy                           0.92      2151
   macro avg       0.46      0.50      0.48      2151
weighted avg       0.84      0.92      0.88      2151

Confusion Matrix:
[[1975    0]
 [ 176    0]]

TE

---
# LightGBM – Original vs SMOTE vs ADASYN (Top-50 Features)

Compare three imbalance-handling strategies, all using the same top-50 filtered features:
- **Original** — `scale_pos_weight` (no oversampling)
- **SMOTE** — Synthetic Minority Over-sampling Technique
- **ADASYN** — Adaptive Synthetic Sampling

In [77]:
from imblearn.over_sampling import SMOTE, ADASYN

# ── SMOTE ─────────────────────────────────────────────────────────
_smote = SMOTE(sampling_strategy='minority', random_state=42, k_neighbors=5)
X_train_smote, y_train_smote = _smote.fit_resample(X_train, y_train)

print(f'Before SMOTE : {np.bincount(y_train)}')
print(f'After  SMOTE : {np.bincount(y_train_smote)}')

# ── ADASYN ────────────────────────────────────────────────────────
_adasyn = ADASYN(sampling_strategy='minority', random_state=42, n_neighbors=5)
X_train_ada, y_train_ada = _adasyn.fit_resample(X_train, y_train)

print(f'Before ADASYN: {np.bincount(y_train)}')
print(f'After  ADASYN: {np.bincount(y_train_ada)}')

Before SMOTE : [5924  528]
After  SMOTE : [5924 5924]
Before ADASYN: [5924  528]
After  ADASYN: [5924 5941]


In [78]:
# ── Train LightGBM (SMOTE, no scale_pos_weight) ───────────────────
train_data_smote = lgb.Dataset(X_train_smote, label=y_train_smote, feature_name=available_features)
val_data_smote   = lgb.Dataset(X_val, label=y_val, feature_name=available_features,
                                reference=train_data_smote)

params_smote = {
    'objective': 'binary',
    'metric': ['auc', 'binary_logloss'],
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'verbose': -1,
    'seed': 42,
}

print('Training LightGBM (SMOTE)...')
_t0_smote = time.time()
evals_result_smote = {}

model_smote = lgb.train(
    params_smote,
    train_data_smote,
    num_boost_round=1000,
    valid_sets=[train_data_smote, val_data_smote],
    valid_names=['train', 'val'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(period=50),
        lgb.record_evaluation(evals_result_smote),
    ],
)

training_time_smote = time.time() - _t0_smote
print(f'\nTraining completed in {training_time_smote:.2f}s')
print(f'Best iteration: {model_smote.best_iteration}')
print(f'Best val AUC: {model_smote.best_score["val"]["auc"]:.4f}')

Training LightGBM (SMOTE)...
Training until validation scores don't improve for 50 rounds
[50]	train's auc: 0.98483	train's binary_logloss: 0.231563	val's auc: 0.732798	val's binary_logloss: 0.327221
[100]	train's auc: 0.994107	train's binary_logloss: 0.139965	val's auc: 0.747563	val's binary_logloss: 0.278268
[150]	train's auc: 0.998027	train's binary_logloss: 0.0981234	val's auc: 0.750768	val's binary_logloss: 0.267204
Early stopping, best iteration is:
[142]	train's auc: 0.997636	train's binary_logloss: 0.10353	val's auc: 0.750834	val's binary_logloss: 0.268244

Training completed in 2.24s
Best iteration: 142
Best val AUC: 0.7508


In [79]:
# ── Evaluate SMOTE model on ORIGINAL (non-oversampled) splits ────
train_preds_smote, train_auc_smote, train_ap_smote, train_time_smote = evaluate(
    model_smote, X_train, y_train, 'TRAIN (SMOTE)')
val_preds_smote, val_auc_smote, val_ap_smote, val_time_smote = evaluate(
    model_smote, X_val, y_val, 'VALIDATION (SMOTE)')
test_preds_smote, test_auc_smote, test_ap_smote, test_time_smote = evaluate(
    model_smote, X_test, y_test, 'TEST (SMOTE)')


TRAIN (SMOTE) Results
ROC-AUC: 0.9776
Average Precision: 0.8396
Scoring Time: 0.0050 seconds

Classification Report:
              precision    recall  f1-score   support

       No DQ       0.96      0.99      0.98      5924
          DQ       0.87      0.58      0.70       528

    accuracy                           0.96      6452
   macro avg       0.92      0.79      0.84      6452
weighted avg       0.96      0.96      0.95      6452

Confusion Matrix:
[[5877   47]
 [ 221  307]]

VALIDATION (SMOTE) Results
ROC-AUC: 0.7508
Average Precision: 0.2184
Scoring Time: 0.0042 seconds

Classification Report:
              precision    recall  f1-score   support

       No DQ       0.93      0.97      0.95      1975
          DQ       0.29      0.13      0.18       176

    accuracy                           0.90      2151
   macro avg       0.61      0.55      0.56      2151
weighted avg       0.87      0.90      0.89      2151

Confusion Matrix:
[[1919   56]
 [ 153   23]]

TEST (SMOTE) R

In [80]:
# ── Train LightGBM (ADASYN, no scale_pos_weight) ─────────────────
train_data_ada = lgb.Dataset(X_train_ada, label=y_train_ada, feature_name=available_features)
val_data_ada   = lgb.Dataset(X_val, label=y_val, feature_name=available_features,
                              reference=train_data_ada)

params_ada = {
    'objective': 'binary',
    'metric': ['auc', 'binary_logloss'],
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'verbose': -1,
    'seed': 42,
}

print('Training LightGBM (ADASYN)...')
_t0_ada = time.time()
evals_result_ada = {}

model_ada = lgb.train(
    params_ada,
    train_data_ada,
    num_boost_round=1000,
    valid_sets=[train_data_ada, val_data_ada],
    valid_names=['train', 'val'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(period=50),
        lgb.record_evaluation(evals_result_ada),
    ],
)

training_time_ada = time.time() - _t0_ada
print(f'\nTraining completed in {training_time_ada:.2f}s')
print(f'Best iteration: {model_ada.best_iteration}')
print(f'Best val AUC: {model_ada.best_score["val"]["auc"]:.4f}')

Training LightGBM (ADASYN)...
Training until validation scores don't improve for 50 rounds
[50]	train's auc: 0.985442	train's binary_logloss: 0.234382	val's auc: 0.729714	val's binary_logloss: 0.330917
[100]	train's auc: 0.993961	train's binary_logloss: 0.141557	val's auc: 0.744146	val's binary_logloss: 0.280969
[150]	train's auc: 0.997826	train's binary_logloss: 0.0993392	val's auc: 0.747585	val's binary_logloss: 0.26999
[200]	train's auc: 0.999319	train's binary_logloss: 0.0736325	val's auc: 0.749276	val's binary_logloss: 0.26722
Early stopping, best iteration is:
[173]	train's auc: 0.998725	train's binary_logloss: 0.0862034	val's auc: 0.750735	val's binary_logloss: 0.267163

Training completed in 2.42s
Best iteration: 173
Best val AUC: 0.7507


In [81]:
# ── Evaluate ADASYN model on ORIGINAL (non-oversampled) splits ────
train_preds_ada, train_auc_ada, train_ap_ada, train_time_ada = evaluate(
    model_ada, X_train, y_train, 'TRAIN (ADASYN)')
val_preds_ada, val_auc_ada, val_ap_ada, val_time_ada = evaluate(
    model_ada, X_val, y_val, 'VALIDATION (ADASYN)')
test_preds_ada, test_auc_ada, test_ap_ada, test_time_ada = evaluate(
    model_ada, X_test, y_test, 'TEST (ADASYN)')


TRAIN (ADASYN) Results
ROC-AUC: 0.9885
Average Precision: 0.9186
Scoring Time: 0.0085 seconds

Classification Report:
              precision    recall  f1-score   support

       No DQ       0.97      1.00      0.98      5924
          DQ       0.94      0.68      0.79       528

    accuracy                           0.97      6452
   macro avg       0.96      0.84      0.89      6452
weighted avg       0.97      0.97      0.97      6452

Confusion Matrix:
[[5903   21]
 [ 171  357]]

VALIDATION (ADASYN) Results
ROC-AUC: 0.7507
Average Precision: 0.2116
Scoring Time: 0.0052 seconds

Classification Report:
              precision    recall  f1-score   support

       No DQ       0.93      0.98      0.95      1975
          DQ       0.29      0.11      0.16       176

    accuracy                           0.90      2151
   macro avg       0.61      0.54      0.56      2151
weighted avg       0.87      0.90      0.89      2151

Confusion Matrix:
[[1926   49]
 [ 156   20]]

TEST (ADASYN

In [82]:
# ── Side-by-side comparison: Original vs SMOTE vs ADASYN ─────────
print('LightGBM – Original vs SMOTE vs ADASYN  (Top-50 filtered features)')
print('=' * 76)
print(f'{"Split":<8} {"Metric":<20} {"Original (spw)":<18} {"SMOTE":<14} {"ADASYN":<10}')
print('-' * 76)

for _split, _spw_auc, _spw_ap, _smt_auc, _smt_ap, _ada_auc, _ada_ap in [
    ('Train', train_auc, train_ap, train_auc_smote, train_ap_smote, train_auc_ada, train_ap_ada),
    ('Val',   val_auc,   val_ap,   val_auc_smote,   val_ap_smote,   val_auc_ada,   val_ap_ada),
    ('Test',  test_auc,  test_ap,  test_auc_smote,  test_ap_smote,  test_auc_ada,  test_ap_ada),
]:
    print(f'{_split:<8} {"ROC-AUC":<20} {_spw_auc:<18.4f} {_smt_auc:<14.4f} {_ada_auc:<10.4f}')
    print(f'{"":8} {"Avg Precision":<20} {_spw_ap:<18.4f} {_smt_ap:<14.4f} {_ada_ap:<10.4f}')

print('=' * 76)
print(f'{"Train time(s)":<28} {training_time:<18.2f} {training_time_smote:<14.2f} {training_time_ada:<10.2f}')

LightGBM – Original vs SMOTE vs ADASYN  (Top-50 filtered features)
Split    Metric               Original (spw)     SMOTE          ADASYN    
----------------------------------------------------------------------------
Train    ROC-AUC              0.8701             0.9776         0.9885    
         Avg Precision        0.3071             0.8396         0.9186    
Val      ROC-AUC              0.7533             0.7508         0.7507    
         Avg Precision        0.2187             0.2184         0.2116    
Test     ROC-AUC              0.7467             0.7928         0.7954    
         Avg Precision        0.2119             0.2707         0.2691    
Train time(s)                0.50               2.24           2.42      


---
# LightGBM – Filtered vs Original Features (ADASYN, Top-50)

Train on the scoring-exclusion-filtered population (`filtered_features_consensus_ordered.csv`) and compare ROC-AUC against the original full population.

In [83]:
# ── Load filtered features ────────────────────────────────────────
X_train_flt, X_val_flt, X_test_flt, y_train_flt, y_val_flt, y_test_flt, feats_flt, _ = \
    load_and_split(n_features=N_FEATURES, features_filename="filtered_features_consensus_ordered.csv")

# Apply ADASYN
_adasyn_flt = ADASYN(sampling_strategy='minority', random_state=42, n_neighbors=5)
X_train_flt_ada, y_train_flt_ada = _adasyn_flt.fit_resample(X_train_flt, y_train_flt)
print(f'Filtered – Before ADASYN: {np.bincount(y_train_flt)}')
print(f'Filtered – After  ADASYN: {np.bincount(y_train_flt_ada)}')

Features  : 50 (top-50)
Samples   : 10754  |  DQ rate: 8.18%
Train     : 6452  (8.18% positive)
Val       : 2151  (8.18% positive)
Test      : 2151  (8.18% positive)
Filtered – Before ADASYN: [5924  528]
Filtered – After  ADASYN: [5924 5941]


In [84]:
# ── Train LightGBM on filtered features (ADASYN) ─────────────────
_td_flt = lgb.Dataset(X_train_flt_ada, label=y_train_flt_ada, feature_name=feats_flt)
_vd_flt = lgb.Dataset(X_val_flt, label=y_val_flt, feature_name=feats_flt, reference=_td_flt)

_params_flt = {**params_ada}  # same hyperparameters

print('Training LightGBM (filtered, ADASYN)...')
_t0_flt = time.time()
_er_flt = {}

model_flt = lgb.train(
    _params_flt, _td_flt,
    num_boost_round=1000,
    valid_sets=[_td_flt, _vd_flt],
    valid_names=['train', 'val'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(period=50),
        lgb.record_evaluation(_er_flt),
    ],
)

training_time_flt = time.time() - _t0_flt
print(f'\nTraining completed in {training_time_flt:.2f}s')
print(f'Best iteration: {model_flt.best_iteration}')
print(f'Best val AUC: {model_flt.best_score["val"]["auc"]:.4f}')

Training LightGBM (filtered, ADASYN)...
Training until validation scores don't improve for 50 rounds
[50]	train's auc: 0.985442	train's binary_logloss: 0.234382	val's auc: 0.729714	val's binary_logloss: 0.330917
[100]	train's auc: 0.993961	train's binary_logloss: 0.141557	val's auc: 0.744146	val's binary_logloss: 0.280969
[150]	train's auc: 0.997826	train's binary_logloss: 0.0993392	val's auc: 0.747585	val's binary_logloss: 0.26999
[200]	train's auc: 0.999319	train's binary_logloss: 0.0736325	val's auc: 0.749276	val's binary_logloss: 0.26722
Early stopping, best iteration is:
[173]	train's auc: 0.998725	train's binary_logloss: 0.0862034	val's auc: 0.750735	val's binary_logloss: 0.267163

Training completed in 2.52s
Best iteration: 173
Best val AUC: 0.7507


In [85]:
# ── Evaluate filtered model ───────────────────────────────────────
_, train_auc_flt, _, _ = evaluate(model_flt, X_train_flt, y_train_flt, 'TRAIN (filtered, ADASYN)')
_, val_auc_flt,   _, _ = evaluate(model_flt, X_val_flt,   y_val_flt,   'VALIDATION (filtered, ADASYN)')
_, test_auc_flt,  _, _ = evaluate(model_flt, X_test_flt,  y_test_flt,  'TEST (filtered, ADASYN)')

# ── Final comparison ──────────────────────────────────────────────
print('\nOriginal (scale_pos_weight)  vs  Original (ADASYN)  vs  Filtered (ADASYN)  — Top-50')
print('=' * 76)
print(f'{"Split":<8} {"Original (spw)":<20} {"Original (ADASYN)":<22} {"Filtered (ADASYN)":<20}')
print('-' * 76)
for _split, _spw, _ada, _flt in [
    ('Train', train_auc,     train_auc_ada, train_auc_flt),
    ('Val',   val_auc,       val_auc_ada,   val_auc_flt),
    ('Test',  test_auc,      test_auc_ada,  test_auc_flt),
]:
    print(f'{_split:<8} {_spw:<20.4f} {_ada:<22.4f} {_flt:<20.4f}')


TRAIN (filtered, ADASYN) Results
ROC-AUC: 0.9885
Average Precision: 0.9186
Scoring Time: 0.0061 seconds

Classification Report:
              precision    recall  f1-score   support

       No DQ       0.97      1.00      0.98      5924
          DQ       0.94      0.68      0.79       528

    accuracy                           0.97      6452
   macro avg       0.96      0.84      0.89      6452
weighted avg       0.97      0.97      0.97      6452

Confusion Matrix:
[[5903   21]
 [ 171  357]]

VALIDATION (filtered, ADASYN) Results
ROC-AUC: 0.7507
Average Precision: 0.2116
Scoring Time: 0.0050 seconds

Classification Report:
              precision    recall  f1-score   support

       No DQ       0.93      0.98      0.95      1975
          DQ       0.29      0.11      0.16       176

    accuracy                           0.90      2151
   macro avg       0.61      0.54      0.56      2151
weighted avg       0.87      0.90      0.89      2151

Confusion Matrix:
[[1926   49]
 [ 156 